# SAM 3 pixel-annotation pilot for CMZ

This notebook tests model-assisted pixel annotation of civets in one daytime frame and one infrared frame. It compares original RGB input with three-channel grayscale input. SAM 3 outputs are proposals: every mask must be visually reviewed before it is accepted as training ground truth.

Outputs are written only to the private Google Drive project folder; footage and masks must not be committed to the public repository.

## 1. Runtime and prerequisites

Use a GPU runtime. Add a Colab secret named `HF_TOKEN`, using a Hugging Face read token from an account approved for `facebook/sam3`.

In [ ]:
import sys, subprocess
from pathlib import Path
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
SAM3_DIR = Path('/content/sam3')
if not SAM3_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/facebookresearch/sam3.git', str(SAM3_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{SAM3_DIR}[notebooks]'], check=True)
print('SAM 3 package is installed. If this was the first installation, restart the session once, then continue below.')

If SAM 3 was installed for the first time, select **Runtime > Restart session**, then resume from the next cell. Do not run any global `bfloat16` autocast cell on a Tesla T4.

In [ ]:
import sys, subprocess
from pathlib import Path
import torch
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download

drive.mount('/content/drive')
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN was not found in Colab Secrets.')
login(token=hf_token)

SAM3_DIR = Path('/content/sam3')
PROJECT_ROOT = Path('/content/drive/MyDrive/CMZprojectphase2')
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'sam3_pixel_annotation_pilot'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not SAM3_DIR.exists():
    raise FileNotFoundError('SAM 3 is missing after restart. Run the installation cell first.')
if not RAW_ROOT.exists():
    raise FileNotFoundError(f'Raw-video folder not found: {RAW_ROOT}')
print('Project:', PROJECT_ROOT)
print('Output:', OUTPUT_ROOT)

## 2. Load the official SAM 3 checkpoint

The tokenizer and checkpoint paths are passed explicitly to avoid the `NoneType` path error seen with automatic discovery. FP16 autocast is used during inference because the Tesla T4 does not support the SAM 3 BF16 path correctly.

In [ ]:
import gc, sys
from pathlib import Path
import torch
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

SAM3_DIR = Path('/content/sam3')
if not SAM3_DIR.exists():
    raise FileNotFoundError('SAM 3 is not installed. Run the installation cell first, restart the session, then retry.')

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN was not found in Colab Secrets.')
login(token=hf_token)

BPE_PATH = SAM3_DIR / 'sam3' / 'assets' / 'bpe_simple_vocab_16e6.txt.gz'
if not BPE_PATH.exists():
    raise FileNotFoundError(f'Tokenizer not found: {BPE_PATH}')

CHECKPOINT_PATH = Path(hf_hub_download(
    repo_id='facebook/sam3',
    filename='sam3.pt',
    token=hf_token,
))
print('Checkpoint:', CHECKPOINT_PATH)

if str(SAM3_DIR) not in sys.path:
    sys.path.insert(0, str(SAM3_DIR))
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

gc.collect()
torch.cuda.empty_cache()
sam_model = build_sam3_image_model(
    bpe_path=str(BPE_PATH),
    checkpoint_path=str(CHECKPOINT_PATH),
    load_from_HF=False,
    device='cuda',
    eval_mode=True,
)
sam_processor = Sam3Processor(sam_model, device='cuda', confidence_threshold=0.5)
print('SAM 3 loaded; model dtype:', next(sam_model.parameters()).dtype)

## 3. Reusable helpers

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

def extract_hevc_frame(video_path, second, output_path, preroll_s=60):
    video_path, output_path = Path(video_path), Path(output_path)
    if not video_path.exists():
        raise FileNotFoundError(video_path)
    output_path.unlink(missing_ok=True)
    coarse_start = max(0, second - preroll_s)
    fine_offset = second - coarse_start
    result = subprocess.run([
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'warning',
        '-ss', str(coarse_start), '-i', str(video_path),
        '-ss', str(fine_offset), '-map', '0:v:0', '-frames:v', '1',
        '-update', '1', '-pix_fmt', 'rgb24', str(output_path),
    ], capture_output=True, text=True)
    if result.returncode != 0 or not output_path.exists():
        raise RuntimeError(result.stderr[-3000:])
    return Image.open(output_path).convert('RGB')

def run_sam(image_input, prompt='animal', threshold=0.5):
    sam_processor.set_confidence_threshold(threshold)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        state = sam_processor.set_image(image_input)
        return sam_processor.set_text_prompt(state=state, prompt=prompt)

def to_numpy(value):
    return value.detach().float().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)

def best_mask(output):
    scores = to_numpy(output['scores']).reshape(-1)
    if not len(scores):
        raise RuntimeError('SAM 3 returned no mask at the selected threshold.')
    index = int(scores.argmax())
    mask = np.squeeze(to_numpy(output['masks'])[index]).astype(bool)
    box = to_numpy(output['boxes'])[index].reshape(-1)[:4]
    return mask, float(scores[index]), box, index

def overlay_image(base_image, mask, color=(0, 255, 0), alpha=0.45):
    result = np.asarray(base_image).copy()
    result[mask] = ((1-alpha)*result[mask] + alpha*np.asarray(color)).astype(np.uint8)
    return result

def mask_iou(a, b):
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0

def save_proposal(folder, stem, source_image, mask, metadata):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    source_image.save(folder / f'{stem}.jpg', quality=95)
    Image.fromarray(mask.astype(np.uint8)*255).save(folder / f'{stem}_mask.png')
    Image.fromarray(overlay_image(source_image, mask)).save(folder / f'{stem}_overlay.jpg', quality=95)
    (folder / f'{stem}.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

## 4. Daytime RGB versus grayscale

The selected frame shows a civet eating. The original video is HEVC, so extraction uses a 60-second pre-roll to recover reference frames.

In [ ]:
DAY_VIDEO_ID = 'A03_20260405140327'
DAY_VIDEO_PATH = RAW_ROOT / 'cam3' / f'{DAY_VIDEO_ID}.mp4'
DAY_SECOND = 6350
DAY_FRAME_PATH = Path('/content/day_006350.png')
day_rgb = extract_hevc_frame(DAY_VIDEO_PATH, DAY_SECOND, DAY_FRAME_PATH)
day_gray = day_rgb.convert('L').convert('RGB')
print('Image size:', day_rgb.size, 'contrast/std:', round(float(np.asarray(day_rgb).std()), 2))
display(day_rgb)

In [ ]:
day_rgb_output = run_sam(day_rgb, prompt='animal', threshold=0.5)
day_gray_output = run_sam(day_gray, prompt='animal', threshold=0.5)
day_rgb_mask, day_rgb_score, day_rgb_box, _ = best_mask(day_rgb_output)
day_gray_mask, day_gray_score, day_gray_box, _ = best_mask(day_gray_output)
day_iou = mask_iou(day_rgb_mask, day_gray_mask)
print('Day RGB score:', round(day_rgb_score, 4))
print('Day grayscale score:', round(day_gray_score, 4))
print('Day RGB-vs-grayscale IoU:', round(day_iou, 4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].imshow(day_rgb); axes[0].set_title('Day source')
axes[1].imshow(overlay_image(day_rgb, day_rgb_mask)); axes[1].set_title(f'RGB mask - {day_rgb_score:.4f}')
axes[2].imshow(overlay_image(day_gray, day_gray_mask)); axes[2].set_title(f'Grayscale mask - {day_gray_score:.4f}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print('Review head, legs, tail, tray, plant and shadows before accepting either mask.')

After visual review, set `DAY_REVIEW_STATUS` to `accepted` only if the grayscale mask follows the visible civet boundary. Otherwise leave it as `needs_review`.

In [ ]:
DAY_REVIEW_STATUS = 'accepted'  # accepted | needs_review | rejected
day_folder = OUTPUT_ROOT / DAY_VIDEO_ID
day_metadata = {
    'video_id': DAY_VIDEO_ID, 'frame_second': DAY_SECOND, 'lighting': 'day',
    'input_mode': 'grayscale_3_channel', 'prompt': 'animal', 'threshold': 0.5,
    'score': day_gray_score, 'box_xyxy': [float(x) for x in day_gray_box],
    'behavior': 'Eating', 'mask_status': DAY_REVIEW_STATUS,
    'rgb_score': day_rgb_score, 'rgb_grayscale_iou': day_iou,
}
save_proposal(day_folder, f'frame_{DAY_SECOND:06d}_gray', day_gray, day_gray_mask, day_metadata)
print('Saved daytime proposal:', day_folder)

## 5. Infrared RGB versus grayscale

The night recording is already nearly grayscale, so identical confidence and masks are expected. This is still checked rather than assumed.

In [ ]:
NIGHT_VIDEO_ID = 'A03_20260405000000'
NIGHT_VIDEO_PATH = RAW_ROOT / 'cam3' / f'{NIGHT_VIDEO_ID}.mp4'
NIGHT_SECOND = 2150
NIGHT_FRAME_PATH = Path('/content/night_002150.png')
night_rgb = extract_hevc_frame(NIGHT_VIDEO_PATH, NIGHT_SECOND, NIGHT_FRAME_PATH)
night_gray = night_rgb.convert('L').convert('RGB')
display(night_rgb)

In [ ]:
night_array = np.asarray(night_rgb).astype(np.int16)
r, g, b = night_array[:, :, 0], night_array[:, :, 1], night_array[:, :, 2]
print('Mean |R-G|:', float(np.abs(r-g).mean()))
print('Mean |R-B|:', float(np.abs(r-b).mean()))
print('Mean |G-B|:', float(np.abs(g-b).mean()))

night_rgb_output = run_sam(night_rgb, prompt='animal', threshold=0.5)
night_gray_output = run_sam(night_gray, prompt='animal', threshold=0.5)
night_rgb_mask, night_rgb_score, night_rgb_box, _ = best_mask(night_rgb_output)
night_gray_mask, night_gray_score, night_gray_box, _ = best_mask(night_gray_output)
night_iou = mask_iou(night_rgb_mask, night_gray_mask)
print('Night RGB score:', round(night_rgb_score, 4))
print('Night grayscale score:', round(night_gray_score, 4))
print('Night RGB-vs-grayscale IoU:', round(night_iou, 4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].imshow(night_rgb); axes[0].set_title('Night source')
axes[1].imshow(overlay_image(night_rgb, night_rgb_mask)); axes[1].set_title(f'Night original - {night_rgb_score:.4f}')
axes[2].imshow(overlay_image(night_gray, night_gray_mask)); axes[2].set_title(f'Night grayscale - {night_gray_score:.4f}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print('Check whether the visible tail and lower body are missing. Confidence is not boundary accuracy.')

The first night text-prompt mask may omit part of the tail. It must therefore remain `needs_review`. The next optional cell uses a positive box prompt for refinement. Adjust the pixel coordinates until the yellow box contains the entire visible civet, including its tail.

In [ ]:
import matplotlib.patches as patches
NIGHT_X1, NIGHT_Y1, NIGHT_X2, NIGHT_Y2 = 620, 240, 930, 800
fig, ax = plt.subplots(figsize=(16, 9)); ax.imshow(night_gray)
ax.add_patch(patches.Rectangle((NIGHT_X1, NIGHT_Y1), NIGHT_X2-NIGHT_X1, NIGHT_Y2-NIGHT_Y1, linewidth=3, edgecolor='yellow', facecolor='none'))
ax.set_title('Adjust this box to include the whole visible civet and tail'); ax.axis('off'); plt.show()

In [ ]:
w, h = night_gray.size
box_cxcywh = [((NIGHT_X1+NIGHT_X2)/2)/w, ((NIGHT_Y1+NIGHT_Y2)/2)/h, (NIGHT_X2-NIGHT_X1)/w, (NIGHT_Y2-NIGHT_Y1)/h]
sam_processor.set_confidence_threshold(0.1)
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    night_box_state = sam_processor.set_image(night_gray)
    night_box_output = sam_processor.add_geometric_prompt(box=box_cxcywh, label=True, state=night_box_state)
night_box_mask, night_box_score, night_box, _ = best_mask(night_box_output)
plt.figure(figsize=(16, 9)); plt.imshow(overlay_image(night_gray, night_box_mask)); plt.title(f'Night box-prompt proposal - {night_box_score:.4f}'); plt.axis('off'); plt.show()

Set `NIGHT_REVIEW_STATUS` to `accepted` only after the refined mask includes the full visible animal and excludes the tray, plant, floor and shadow. Otherwise keep `needs_review` and correct it in CVAT.

In [ ]:
NIGHT_REVIEW_STATUS = 'needs_review'  # change only after visual verification
night_folder = OUTPUT_ROOT / NIGHT_VIDEO_ID
night_metadata = {
    'video_id': NIGHT_VIDEO_ID, 'frame_second': NIGHT_SECOND, 'lighting': 'ir_night',
    'input_mode': 'grayscale_3_channel', 'prompt': 'positive_box', 'threshold': 0.1,
    'score': night_box_score, 'box_xyxy': [float(x) for x in night_box],
    'behavior': 'Eating', 'mask_status': NIGHT_REVIEW_STATUS,
    'text_prompt_score': night_gray_score, 'rgb_grayscale_iou': night_iou,
}
save_proposal(night_folder, f'frame_{NIGHT_SECOND:06d}_gray_box', night_gray, night_box_mask, night_metadata)
print('Saved night proposal:', night_folder)

## 6. Pilot summary

This table records model proposals, not segmentation accuracy. True IoU or Dice requires human-corrected ground-truth masks.

In [ ]:
summary = pd.DataFrame([
    {'video_id': DAY_VIDEO_ID, 'second': DAY_SECOND, 'lighting': 'day', 'rgb_score': day_rgb_score, 'gray_score': day_gray_score, 'rgb_gray_iou': day_iou, 'mask_status': DAY_REVIEW_STATUS},
    {'video_id': NIGHT_VIDEO_ID, 'second': NIGHT_SECOND, 'lighting': 'ir_night', 'rgb_score': night_rgb_score, 'gray_score': night_gray_score, 'rgb_gray_iou': night_iou, 'mask_status': NIGHT_REVIEW_STATUS},
])
summary_path = OUTPUT_ROOT / 'pilot_summary.csv'
summary.to_csv(summary_path, index=False)
display(summary)
print('Saved:', summary_path)

## Interpretation and next step

For the tested daytime frame, grayscale preserved the mask and slightly increased confidence; for the infrared frame, original and grayscale results were identical because IR footage was already effectively grayscale. These two frames are a pipeline pilot, not enough evidence to claim grayscale is universally better.

Next, sample a balanced review set across cameras, day/night and behaviors. Use SAM 3 to propose masks, correct them in CVAT, export reviewed segmentation annotations, and only then train/evaluate a segmentation model. Split data by video, never by random frame.